# 01 — Macroeconomic Surprise Construction

## BlackRock Project
### State-Dependent Intraday Response of S&P 500 Futures to Macroeconomic Surprises

### Objective

This notebook constructs clean macroeconomic surprise measures from Bloomberg economic-release data.

The main raw surprise measure is:

\[$
\text{Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Consensus}_{i,t}$
\]

Because different macroeconomic indicators use different units, raw surprises will later be converted into **causal standardized surprises** using only information available before each release.

### Initial macroeconomic events

- CPI
- Core CPI
- Nonfarm Payrolls
- Unemployment Rate
- Average Hourly Earnings
- PCE
- Core PCE

### Planned output

The final processed dataset will be saved as:

`data/processed/macro_surprises_2010_2026.parquet`

### Research workflow

1. Load and inspect the Bloomberg workbook.
2. Standardize column names and combine yearly sheets.
3. Identify the relevant macroeconomic releases.
4. Convert Bloomberg values into numeric fields.
5. Construct `actual - consensus` surprise measures.
6. Build causal standardized surprise variables.
7. Validate event timestamps and missing observations.
8. Save a clean processed dataset for the event-response analysis.


## Notebook Workflow

This notebook transforms raw Bloomberg economic-release data into a clean,
research-ready dataset of macroeconomic surprises for the project:

**State-Dependent Intraday Response of S&P 500 Futures to Macroeconomic Surprises**

The code is organized into eight main parts.

### Step 1 — Set Up the Environment and Project Paths

The first section imports the required Python packages and defines the project
directories.

The notebook locates:

- the raw Bloomberg workbook,
- the processed-data directory,
- and the project root.

This ensures that all input and output files are referenced consistently
throughout the analysis.

---

### Step 2 — Inspect the Bloomberg Workbook

The Bloomberg workbook contains separate yearly sheets covering 2010–2026.

This section checks:

- the number of sheets,
- sheet names,
- workbook coverage,
- and basic file structure.

The purpose is to understand the source before modifying the data.

---

### Step 3 — Inspect the Raw Bloomberg Data

One yearly sheet is examined in detail before combining all years.

This section identifies:

- original Bloomberg column names,
- data types,
- release dates,
- release times,
- event names,
- actual values,
- Bloomberg consensus forecasts,
- prior values,
- revisions,
- scaling factors,
- and Bloomberg tickers.

This inspection determines how the raw Bloomberg export should be cleaned.

---

### Step 4 — Standardize and Combine All Yearly Sheets

The long Bloomberg formula-based column names are renamed to concise,
research-friendly variables.

The 17 yearly sheets are then combined into one master dataset.

The resulting dataset contains:

\[
\boxed{28,159}
\]

U.S. economic-release observations covering January 2010 through July 2026.

This section also validates:

- yearly observation counts,
- missing values,
- country coverage,
- and Bloomberg series identifiers.

---

### Step 5 — Construct and Evaluate Macroeconomic Surprises

For releases with both an actual value and a Bloomberg consensus forecast, the
raw surprise is defined as:

\[$
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Consensus}_{i,t}.$
\]

Because different indicators use different units, surprises are also
standardized relative to their own historical forecast-error distributions.

The standardized surprise is based only on information available before the
current release:

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
}.$
\]

This causal construction prevents look-ahead bias.

The section also examines unusually large macroeconomic surprises, including
extreme COVID-era labor-market shocks.

---

### Step 6 — Select the Final Macroeconomic Series

The full Bloomberg calendar contains thousands of economic indicators, but the
main project focuses on seven components from three major event families.

#### CPI

- Headline CPI MoM
- Core CPI MoM

#### NFP / Labor Market

- Change in Nonfarm Payrolls
- Unemployment Rate
- Average Hourly Earnings MoM

#### PCE

- Headline PCE Price Index MoM
- Core PCE Price Index MoM

The exact Bloomberg tickers are used to identify these series consistently.

After filtering, the final macroeconomic sample contains:
\[$
\boxed{1,359}$
\]

release-component observations.

A minimum of 24 previous valid releases is required to construct the final
causal standardized surprise.

Special simultaneous multi-period announcements are also handled so that
observations released at the same timestamp cannot be used to standardize one
another.

---

### Step 7 — Construct Final Event Timestamps and Save the Dataset

Bloomberg release dates and release times are combined into timezone-aware
announcement timestamps.

Release times are first interpreted using:

`America/New_York`

and then converted to UTC.

This automatically handles the difference between Eastern Standard Time and
Eastern Daylight Time.

The final dataset contains:

- Eastern Time and UTC timestamps,
- event family,
- release component,
- reference period,
- Bloomberg ticker,
- actual value,
- consensus forecast,
- prior value,
- revision,
- raw surprise,
- causal standardized surprise,
- capped standardized surprise,
- and a multi-period-release flag.

After final integrity checks, the dataset is saved as:

`data/processed/macro_surprises_2010_2026.parquet`

This file will be used in the next notebook to align macroeconomic releases with
one-minute E-mini S&P 500 futures data.

---

### Step 8 — Results and Conclusion

The final section summarizes the completed macroeconomic surprise dataset and
checks that the processing pipeline is internally consistent.

The final dataset contains:

\[
\boxed{1,359 \text{ observations}}
\]

with:

- 1,358 valid raw surprises,
- 1,190 valid causal standardized surprises,
- 0 missing Eastern Time timestamps,
- 0 missing UTC timestamps,
- 12 observations belonging to simultaneous multi-period releases,
- 0 exact duplicate rows.

The final sample covers:

\[$
\boxed{
\text{January 2010}
\rightarrow
\text{July 2026}
}$
\]

The output from this notebook provides the macroeconomic input for the next
stage of the project:

\[$
\boxed{
\text{Macro Surprise}
\rightarrow
\text{ES Futures Response}
\rightarrow
\text{Pre-Event Market State}
\rightarrow
\text{State-Dependent Response}
}$
\]

The next notebook will measure E-mini S&P 500 futures responses over
1-, 5-, 15-, 30-, and 60-minute horizons following each macroeconomic
announcement.

## 1. Imports and project paths

This section loads the required Python packages and defines the repository paths.

The notebook is expected to be stored in:

`notebooks/01_macro_surprise_cleaning.ipynb`

Therefore, the project root should be one directory above the notebook folder.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

BLOOMBERG_FILE = RAW_DIR / "Bloomberg Economic Releases.xlsx"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Bloomberg file:", BLOOMBERG_FILE)
print("Bloomberg file exists:", BLOOMBERG_FILE.exists())


Project root: /Users/reza/Desktop/mfe-term 3/BlackRock-Project-State-Dependent-Intraday-Response-of-S-P-500-Futures-to-Macroeconomic-Surprises
Raw data directory: /Users/reza/Desktop/mfe-term 3/BlackRock-Project-State-Dependent-Intraday-Response-of-S-P-500-Futures-to-Macroeconomic-Surprises/data/raw
Processed data directory: /Users/reza/Desktop/mfe-term 3/BlackRock-Project-State-Dependent-Intraday-Response-of-S-P-500-Futures-to-Macroeconomic-Surprises/data/processed
Bloomberg file: /Users/reza/Desktop/mfe-term 3/BlackRock-Project-State-Dependent-Intraday-Response-of-S-P-500-Futures-to-Macroeconomic-Surprises/data/raw/Bloomberg Economic Releases.xlsx
Bloomberg file exists: True


### Checkpoint

Before continuing, confirm that the final line above is:

`Bloomberg file exists: True`

If it is `False`, do not continue yet. Check that the Excel file is located at:

`data/raw/Bloomberg Economic Releases.xlsx`


## 2. Inspect the Bloomberg workbook

We first inspect the workbook structure before performing any cleaning.

This lets us verify:

- the number of yearly sheets,
- the sheet names,
- and whether the workbook covers the expected 2010–2026 period.


In [13]:
# ------------------------------------------------------------
# Inspect workbook sheet names
# ------------------------------------------------------------

xls = pd.ExcelFile(BLOOMBERG_FILE)

print("Number of sheets:", len(xls.sheet_names))
print("\nSheet names:")
print(xls.sheet_names)


Number of sheets: 17

Sheet names:
['2026', '2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010']


## 3. Inspect one raw sheet

Before combining all years, inspect one sheet exactly as Bloomberg stores it.

We want to identify:

- the raw column names,
- the data types,
- how dates and times are represented,
- how missing values appear,
- and how `ACTUAL`, `SURVEY_MEDIAN`, `PRIOR`, and `REVISION` are stored.


In [21]:
# ------------------------------------------------------------
# Read one sample sheet
# ------------------------------------------------------------

sample_sheet = xls.sheet_names[0]

sample = pd.read_excel(
    BLOOMBERG_FILE,
    sheet_name=sample_sheet
)

print("Sample sheet:", sample_sheet)
print("Shape:", sample.shape)

display(sample.head(10))
#display(sample[[EVENT_NAME =='ISM Manufacturing']].head(10))


print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

Sample sheet: 2026
Shape: (1241, 12)


,ID,"DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).COUNTRY_NAME","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).RELEASE_DATE","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).RELEASE_TIME","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).EVENT_NAME","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).PERIOD","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).SURVEY_MEDIAN","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).ACTUAL","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).PRIOR","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).REVISION","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).SCALING_FACTOR","DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1)"
0,US Country,United States,2026-01-02,09:45:00,S&P Global US Manufacturing PMI,Dec F,51.80,51.80,52.2,51.8,NaN,MPMIUSMA Index
1,US Country,United States,2026-01-05,10:00:00,ISM New Orders,Dec,NaN,47.70,47.4,47.3,NaN,NAPMNEWO Index
2,US Country,United States,2026-01-05,10:00:00,ISM Manufacturing,Dec,48.44,47.90,48.2,48.0,NaN,NAPMPMI Index
3,US Country,United States,2026-01-05,10:00:00,ISM Employment,Dec,NaN,44.90,44.0,44.1,NaN,NAPMEMPL Index
4,US Country,United States,2026-01-05,10:00:00,ISM Prices Paid,Dec,58.70,58.50,58.5,NaN,NaN,NAPMPRIC Index
5,US Country,United States,2026-01-05,17:28:00,Omdia Total Vehicle Sales,Dec,15.80,16.02,15.6,NaN,m,SAARTOTL Index
6,US Country,United States,2026-01-06,09:45:00,S&P Global US Services PMI,Dec F,52.90,52.50,54.1,52.5,NaN,MPMIUSSA Index
7,US Country,United States,2026-01-06,09:45:00,S&P Global US Composite PMI,Dec F,NaN,52.70,54.2,52.7,NaN,MPMIUSCA Index
8,US Country,United States,2026-01-07,07:00:00,MBA Releases Dec. 26 and Jan. 2 Week Ending Re...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,US Country,United States,2026-01-07,07:00:00,MBA Mortgage Applications,2026-12-26 00:00:00,NaN,-10.00,-5.0,NaN,%,MBAVCHNG Index



Columns:
['ID', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).COUNTRY_NAME', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).RELEASE_DATE', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).RELEASE_TIME', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).EVENT_NAME', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).PERIOD', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).SURVEY_MEDIAN', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).ACTUAL', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).PRIOR', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).REVISION', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIEW=Standard),REMOVE_ID=1).SCALING_FACTOR', 'DROPNA(CALENDAR(TYPE=ECONOMIC_RELEASES,SUBTYPE=ALL,VIE

### Stop here first

At this stage, **do not clean or transform the data yet**.

First review the outputs from Sections 2 and 3.

The next step will be to build a robust loader that combines all yearly sheets while preserving the original Bloomberg fields.


## 4. Standardize Bloomberg columns and combine yearly sheets

The Bloomberg export uses long formula-based column names.

In this section we:

1. Rename the raw Bloomberg columns to concise names.
2. Preserve the Bloomberg series ticker.
3. Add the source year for validation.
4. Combine all yearly sheets into one dataset.
5. Perform basic structural checks.

No macroeconomic filtering is performed yet.

In [22]:
# Helper: clean Bloomberg column names


In [38]:
# ------------------------------------------------------------
# Helper: clean Bloomberg column names
# ------------------------------------------------------------

def clean_bloomberg_columns(df):
    """
    Rename Bloomberg formula-based column names to concise,
    analysis-friendly names while preserving all raw fields.
    """
    
    df = df.copy()

    rename_map = {}

    for col in df.columns:
        
        if col == "ID":
            rename_map[col] = "id"
            
        elif col.endswith(".COUNTRY_NAME"):
            rename_map[col] = "country_name"
            
        elif col.endswith(".RELEASE_DATE"):
            rename_map[col] = "release_date"
            
        elif col.endswith(".RELEASE_TIME"):
            rename_map[col] = "release_time"
            
        elif col.endswith(".EVENT_NAME"):
            rename_map[col] = "event_name"
            
        elif col.endswith(".PERIOD"):
            rename_map[col] = "period"
            
        elif col.endswith(".SURVEY_MEDIAN"):
            rename_map[col] = "survey_median"
            
        elif col.endswith(".ACTUAL"):
            rename_map[col] = "actual"
            
        elif col.endswith(".PRIOR"):
            rename_map[col] = "prior"
            
        elif col.endswith(".REVISION"):
            rename_map[col] = "revision"
            
        elif col.endswith(".SCALING_FACTOR"):
            rename_map[col] = "scaling_factor"
            
        else:
            # The final Bloomberg column contains identifiers
            # such as "MPMIUSMA Index", "NAPMPMI Index", etc.
            rename_map[col] = "bloomberg_ticker"

    return df.rename(columns=rename_map)

In [39]:
# Combine all yearly Bloomberg sheets


In [40]:
# ------------------------------------------------------------
# Combine all yearly Bloomberg sheets
# ------------------------------------------------------------

frames = []

for sheet_name in xls.sheet_names:
    
    df_year = pd.read_excel(
        BLOOMBERG_FILE,
        sheet_name=sheet_name
    )
    
    df_year = clean_bloomberg_columns(df_year)
    
    # Preserve the Bloomberg workbook sheet/year
    df_year["source_year"] = int(sheet_name)
    
    frames.append(df_year)

macro_all = pd.concat(
    frames,
    ignore_index=True
)

# Make sure release_date is datetime
macro_all["release_date"] = pd.to_datetime(
    macro_all["release_date"],
    errors="coerce"
)

# Sort chronologically
macro_all = macro_all.sort_values(
    ["release_date", "release_time", "event_name"]
).reset_index(drop=True)

print("Combined shape:", macro_all.shape)

print("\nColumns:")
print(macro_all.columns.tolist())

print("\nDate range:")
print(
    macro_all["release_date"].min(),
    "to",
    macro_all["release_date"].max()
)

print("\nRows by source year:")
display(
    macro_all.groupby("source_year")
    .size()
    .rename("n_rows")
    .to_frame()
)

display(macro_all.head(10))

Combined shape: (28159, 13)

Columns:
['id', 'country_name', 'release_date', 'release_time', 'event_name', 'period', 'survey_median', 'actual', 'prior', 'revision', 'scaling_factor', 'bloomberg_ticker', 'source_year']

Date range:
2010-01-04 00:00:00 to 2026-07-17 00:00:00

Rows by source year:


,n_rows
source_year,
2010,1329
2011,1348
2012,1426
2013,1478
2014,1548
2015,1722
2016,1778
2017,1802
2018,1810


,id,country_name,release_date,release_time,event_name,period,survey_median,actual,prior,revision,scaling_factor,bloomberg_ticker,source_year
0,US Country,United States,2010-01-04,10:00:00,Construction Spending MoM,Nov,-0.500,-0.600000,0.00,-1.792150,%,CNSTTMOM Index,2010
1,US Country,United States,2010-01-04,10:00:00,ISM Manufacturing,Dec,54.300,55.900002,53.60,55.400000,NaN,NAPMPMI Index,2010
2,US Country,United States,2010-01-04,10:00:00,ISM Prices Paid,Dec,57.200,61.500000,55.00,55.400000,NaN,NAPMPRIC Index,2010
3,US Country,United States,2010-01-05,10:00:00,Factory Orders,Nov,0.500,1.100000,0.60,1.936030,%,TMNOCHNG Index,2010
4,US Country,United States,2010-01-05,10:00:00,Pending Home Sales MoM,Nov,-2.000,-16.000000,3.70,3.490560,%,USPHTMOM Index,2010
5,US Country,United States,2010-01-05,10:00:00,Pending Home Sales NSA YoY,Nov,31.000,19.300000,28.60,29.312962,%,USPHTYOY Index,2010
6,US Country,United States,2010-01-05,17:00:00,ABC Consumer Confidence,2026-01-03 00:00:00,NaN,-41.000000,-44.00,NaN,NaN,ACNFCOMF Index,2010
7,US Country,United States,2010-01-05,17:00:00,Langer Consumer Comfort,2026-01-03 00:00:00,NaN,-41.000000,-44.00,28.000000,NaN,COMFCOMF Index,2010
8,US Country,United States,2010-01-05,NaN,Omdia Total Vehicle Sales,Dec,11.000,11.230000,10.92,10.850000,m,SAARTOTL Index,2010
9,US Country,United States,2010-01-05,NaN,Wards Domestic Vehicle Sales,Dec,8.355,8.630000,8.36,8.340000,m,SAARDTOT Index,2010


In [41]:
print(macro_all.columns.tolist())

['id', 'country_name', 'release_date', 'release_time', 'event_name', 'period', 'survey_median', 'actual', 'prior', 'revision', 'scaling_factor', 'bloomberg_ticker', 'source_year']


In [42]:
# Then recreate the raw surprise:

In [43]:
macro_all["raw_surprise"] = (
    macro_all["actual"] - macro_all["survey_median"]
)

In [29]:
#basic validation 

In [31]:
# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("Unique countries:")
print(macro_all["country_name"].value_counts(dropna=False).head())

print("\nUnique ID values:")
print(macro_all["id"].value_counts(dropna=False).head())

print("\nMissing values:")
missing_summary = pd.DataFrame({
    "missing_n": macro_all.isna().sum(),
    "missing_pct": 100 * macro_all.isna().mean()
}).sort_values("missing_pct", ascending=False)

display(missing_summary)

print("\nSample Bloomberg tickers:")
display(
    macro_all[
        ["event_name", "bloomberg_ticker"]
    ]
    .drop_duplicates()
    .head(20)
)

Unique countries:
country_name
United States    28159
Name: count, dtype: int64

Unique ID values:
id
US Country    28159
Name: count, dtype: int64

Missing values:


,missing_n,missing_pct
revision,9240,32.813665
scaling_factor,8161,28.981853
survey_median,6587,23.392166
prior,1820,6.463298
actual,1743,6.189850
period,768,2.727370
bloomberg_ticker,768,2.727370
release_time,198,0.703150
id,0,0.000000
country_name,0,0.000000



Sample Bloomberg tickers:


,event_name,bloomberg_ticker
0,Construction Spending MoM,CNSTTMOM Index
1,ISM Manufacturing,NAPMPMI Index
2,ISM Prices Paid,NAPMPRIC Index
3,Factory Orders,TMNOCHNG Index
4,Pending Home Sales MoM,USPHTMOM Index
5,Pending Home Sales NSA YoY,USPHTYOY Index
6,ABC Consumer Confidence,ACNFCOMF Index
7,Langer Consumer Comfort,COMFCOMF Index
8,Omdia Total Vehicle Sales,SAARTOTL Index
9,Wards Domestic Vehicle Sales,SAARDTOT Index


## 5. Construct Raw and Standardized Macroeconomic Surprises

For each economic release, the raw surprise is defined as:

\[$
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Survey Median}_{i,t}$
\]

Because macroeconomic indicators are measured in different units, raw surprises
cannot be compared directly across releases.

For example:

- CPI surprises are measured in percentage points.
- Nonfarm Payroll surprises are measured in thousands of jobs.
- ISM surprises are measured in index points.

We therefore construct a causal standardized surprise:

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
}$
\]

where the historical mean and standard deviation use only observations released
before time \(t\). This avoids look-ahead bias.

In [32]:
# Raw macroeconomic surprise


In [33]:
# ------------------------------------------------------------
# Raw macroeconomic surprise
# ------------------------------------------------------------

macro_all["raw_surprise"] = (
    macro_all["actual"]
    - macro_all["survey_median"]
)

print("Observations:", len(macro_all))
print("Valid raw surprises:", macro_all["raw_surprise"].notna().sum())
print(
    "Missing raw surprises:",
    macro_all["raw_surprise"].isna().sum()
)

display(
    macro_all[
        [
            "release_date",
            "release_time",
            "event_name",
            "bloomberg_ticker",
            "survey_median",
            "actual",
            "raw_surprise",
        ]
    ].head(20)
)

Observations: 28159
Valid raw surprises: 20941
Missing raw surprises: 7218


,release_date,release_time,event_name,bloomberg_ticker,survey_median,actual,raw_surprise
0,2010-01-04,10:00:00,Construction Spending MoM,CNSTTMOM Index,-0.500,-0.600000,-0.100000
1,2010-01-04,10:00:00,ISM Manufacturing,NAPMPMI Index,54.300,55.900002,1.600002
2,2010-01-04,10:00:00,ISM Prices Paid,NAPMPRIC Index,57.200,61.500000,4.300000
3,2010-01-05,10:00:00,Factory Orders,TMNOCHNG Index,0.500,1.100000,0.600000
4,2010-01-05,10:00:00,Pending Home Sales MoM,USPHTMOM Index,-2.000,-16.000000,-14.000000
5,2010-01-05,10:00:00,Pending Home Sales NSA YoY,USPHTYOY Index,31.000,19.300000,-11.700000
6,2010-01-05,17:00:00,ABC Consumer Confidence,ACNFCOMF Index,NaN,-41.000000,NaN
7,2010-01-05,17:00:00,Langer Consumer Comfort,COMFCOMF Index,NaN,-41.000000,NaN
8,2010-01-05,NaN,Omdia Total Vehicle Sales,SAARTOTL Index,11.000,11.230000,0.230000
9,2010-01-05,NaN,Wards Domestic Vehicle Sales,SAARDTOT Index,8.355,8.630000,0.275000


In [34]:
# Causal standardized surprise


In [44]:
# ------------------------------------------------------------
# Causal standardized surprise
# ------------------------------------------------------------

MIN_HISTORY = 12

# Sort chronologically within each Bloomberg series
macro_all = macro_all.sort_values(
    ["bloomberg_ticker", "release_date", "release_time"]
).reset_index(drop=True)

# Historical mean using ONLY prior releases
macro_all["surprise_hist_mean"] = (
    macro_all
    .groupby("bloomberg_ticker", dropna=False)["raw_surprise"]
    .transform(
        lambda x:
        x.expanding(min_periods=MIN_HISTORY)
         .mean()
         .shift(1)
    )
)

# Historical standard deviation using ONLY prior releases
macro_all["surprise_hist_std"] = (
    macro_all
    .groupby("bloomberg_ticker", dropna=False)["raw_surprise"]
    .transform(
        lambda x:
        x.expanding(min_periods=MIN_HISTORY)
         .std()
         .shift(1)
    )
)

# Standardized surprise
macro_all["z_surprise"] = (
    macro_all["raw_surprise"]
    - macro_all["surprise_hist_mean"]
) / macro_all["surprise_hist_std"]

print("Standardization complete.")
print("Columns now:")
print(macro_all.columns.tolist())

Standardization complete.
Columns now:
['id', 'country_name', 'release_date', 'release_time', 'event_name', 'period', 'survey_median', 'actual', 'prior', 'revision', 'scaling_factor', 'bloomberg_ticker', 'source_year', 'raw_surprise', 'surprise_hist_mean', 'surprise_hist_std', 'z_surprise']


In [45]:
# Inspect standardized surprises


In [46]:
# ------------------------------------------------------------
# Inspect standardized surprises
# ------------------------------------------------------------

display(
    macro_all[
        [
            "release_date",
            "event_name",
            "bloomberg_ticker",
            "survey_median",
            "actual",
            "raw_surprise",
            "surprise_hist_mean",
            "surprise_hist_std",
            "z_surprise",
        ]
    ]
    .dropna(subset=["z_surprise"])
    .head(20)
)

,release_date,event_name,bloomberg_ticker,survey_median,actual,raw_surprise,surprise_hist_mean,surprise_hist_std,z_surprise
28,2010-07-20,ABC Consumer Confidence,ACNFCOMF Index,-44.0,-45.0,-1.0,-1.291667,1.912380,0.152515
29,2010-07-27,ABC Consumer Confidence,ACNFCOMF Index,-46.0,-48.0,-2.0,-1.269231,1.832750,-0.398728
30,2010-08-03,ABC Consumer Confidence,ACNFCOMF Index,-46.0,-50.0,-4.0,-1.321429,1.771648,-1.511910
32,2010-08-17,ABC Consumer Confidence,ACNFCOMF Index,-48.0,-45.0,3.0,-1.500000,1.841971,2.443035
33,2010-08-24,ABC Consumer Confidence,ACNFCOMF Index,-46.0,-44.0,2.0,-1.218750,2.105301,1.528879
34,2010-08-31,ABC Consumer Confidence,ACNFCOMF Index,-44.0,-45.0,-1.0,-1.029412,2.182821,0.013474
35,2010-09-07,ABC Consumer Confidence,ACNFCOMF Index,-44.0,-43.0,1.0,-1.027778,2.117658,0.957557
43,2010-11-02,ABC Consumer Confidence,ACNFCOMF Index,-47.0,-46.0,1.0,-0.921053,2.109918,0.910487
47,2010-11-30,ABC Consumer Confidence,ACNFCOMF Index,-46.0,-45.0,1.0,-0.825000,2.098088,0.869840
54,2011-01-18,ABC Consumer Confidence,ACNFCOMF Index,-40.0,-43.0,-3.0,-0.738095,2.083381,-1.085689


### Interpretation of Raw and Standardized Macroeconomic Surprises

For each macroeconomic release, the raw surprise is defined as

\[$
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Survey Median}_{i,t}.$
\]

This measures the unexpected component of the announcement relative to the Bloomberg consensus forecast.

For example, for ISM Manufacturing on January 4, 2010,

\[$
\text{Actual}=55.9,
\qquad
\text{Consensus}=54.3,$
\]

so

\[$
\text{Raw Surprise}
=
55.9-54.3
=
+1.6.$
\]

A positive raw surprise means that the released value was above consensus, while a negative raw surprise means that it was below consensus.

However, raw surprises cannot be compared directly across economic indicators because different releases use different units. For example, CPI surprises are measured in percentage points, payroll surprises are measured in thousands of jobs, and ISM surprises are measured in index points.

To make surprises comparable across releases, each series is standardized using its own historical forecast errors:

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
},$
\]

where \(\mu_{i,<t}\) and \(\sigma_{i,<t}\) are the historical mean and standard deviation of surprises for the same Bloomberg series using only observations that occurred before the current release.

This causal construction is important because it avoids look-ahead bias. The current observation and future releases are not used when calculating the standardized surprise.

For example, suppose a release has

\[$
\text{Raw Surprise}=3.0,$
\]
while the historical mean surprise is
\[
-1.5
\]
and the historical standard deviation is
\[
1.842.
\]
Then

\[$
z
=
\frac{3.0-(-1.5)}{1.842}
\approx
2.44.$
\]

A \(z\)-surprise of approximately \(+2.44\) means that the release was about 2.4 historical standard deviations above its typical forecast error and therefore represents an unusually large positive surprise.

Similarly:

- \(z \approx 0\): the surprise is close to its historical norm.
- \(z \approx +1\): moderately above expectations.
- \(z \approx +2\): unusually above expectations.
- \(z \approx -1\): moderately below expectations.
- \(z \approx -2\): unusually below expectations.

The sign of the surprise should not automatically be interpreted as bullish or bearish for equities. For example, a positive inflation surprise may be negative for S&P 500 futures if it increases expectations of tighter monetary policy, while a negative unemployment-claims surprise may represent stronger labor-market conditions.

Therefore, the project keeps the surprise measure mechanically defined as

\[
\text{Actual}-\text{Consensus},
\]

and later estimates how S&P 500 futures actually respond to each type of macroeconomic surprise.

The resulting workflow is:

\[$
\text{Actual}
-
\text{Consensus}
\rightarrow
\text{Raw Surprise}
\rightarrow
\text{Causal Standardized Surprise}
\rightarrow
\text{Intraday ES Response}.$
\]

At this stage, the standardization confirms that the surprise-construction pipeline is working. The next step is to restrict the analysis to the main macroeconomic releases used in the project, including CPI, Core CPI, Nonfarm Payrolls, Unemployment, Average Hourly Earnings, PCE, and Core PCE.

## 6. Final Macroeconomic Surprise Construction

The previous section verified that raw and causal standardized surprises can be
constructed correctly from the Bloomberg data.

The next step is to restrict the analysis to the major macroeconomic releases
used in this project.

### Primary target releases

The initial analysis focuses on:

- CPI
- Core CPI
- Nonfarm Payrolls
- Unemployment Rate
- Average Hourly Earnings
- PCE Price Index
- Core PCE Price Index

For each component, the raw surprise is

\[$
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Consensus}_{i,t}.$
\]

Because different economic indicators are measured in different units, the raw
surprise is standardized separately for each Bloomberg series:

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
}.$
\]

Only historical observations available before release \(t\) are used when
computing the historical mean and standard deviation. This prevents look-ahead
bias.

Before constructing the final signal, the exact Bloomberg event names and
tickers are identified directly from the data.

In [47]:
#Then add Part 6.1 — Find the exact Bloomberg series:

In [48]:

# ------------------------------------------------------------
# Part 6.1: Identify target macroeconomic series
# ------------------------------------------------------------

search_terms = [
    "CPI",
    "Consumer Price",
    "Nonfarm",
    "Payroll",
    "Unemployment",
    "Hourly",
    "Earnings",
    "PCE",
    "Personal Consumption",
]

pattern = "|".join(search_terms)

target_candidates = (
    macro_all.loc[
        macro_all["event_name"].str.contains(
            pattern,
            case=False,
            na=False
        ),
        [
            "event_name",
            "bloomberg_ticker",
            "scaling_factor",
        ],
    ]
    .drop_duplicates()
    .sort_values(
        ["event_name", "bloomberg_ticker"]
    )
    .reset_index(drop=True)
)

print("Number of candidate Bloomberg series:",
      len(target_candidates))

display(target_candidates)


Number of candidate Bloomberg series: 42


,event_name,bloomberg_ticker,scaling_factor
0,Annual Revisions: CPI,NaN,NaN
1,Average Hourly Earnings MoM,AHE MOM% Index,%
2,Average Hourly Earnings YoY,AHE YOY% Index,%
3,Avg Hourly Earning MOM Prod,USHETOT% Index,%
4,Avg Hourly Earning YOY Prod,USHEYOY Index,%
5,BLS Will Not Publish Oct. All-Items and Core CPI,NaN,NaN
6,BLS Will Not Publish October Real Earnings Data,NaN,NaN
7,BLS preliminary annual payrolls benchmark revi...,NaN,NaN
8,BLS to Release Oct.-Nov. Jobs Data; No Oct. Un...,NaN,NaN
9,CPI Index NSA,CPURNSA Index,NaN


In [49]:
# Part 6.2: Candidate series coverage


In [50]:
# ------------------------------------------------------------
# Part 6.2: Candidate series coverage
# ------------------------------------------------------------

candidate_coverage = (
    macro_all[
        macro_all["event_name"].isin(
            target_candidates["event_name"]
        )
    ]
    .groupby(
        ["event_name", "bloomberg_ticker"],
        dropna=False
    )
    .agg(
        n_observations=("event_name", "size"),
        first_release=("release_date", "min"),
        last_release=("release_date", "max"),
        n_actual=("actual", "count"),
        n_consensus=("survey_median", "count"),
        n_raw_surprise=("raw_surprise", "count"),
    )
    .reset_index()
    .sort_values(
        ["event_name", "first_release"]
    )
)

display(candidate_coverage)

,event_name,bloomberg_ticker,n_observations,first_release,last_release,n_actual,n_consensus,n_raw_surprise
0,Annual Revisions: CPI,NaN,2,2025-02-12,2026-02-13,0,0,0
1,Average Hourly Earnings MoM,AHE MOM% Index,198,2010-02-05,2026-07-02,198,197,197
2,Average Hourly Earnings YoY,AHE YOY% Index,198,2010-02-05,2026-07-02,198,197,197
3,Avg Hourly Earning MOM Prod,USHETOT% Index,2,2010-01-08,2010-02-05,2,2,2
4,Avg Hourly Earning YOY Prod,USHEYOY Index,2,2010-01-08,2010-02-05,2,2,2
5,BLS Will Not Publish Oct. All-Items and Core CPI,NaN,1,2025-12-18,2025-12-18,0,0,0
6,BLS Will Not Publish October Real Earnings Data,NaN,1,2025-12-18,2025-12-18,0,0,0
7,BLS preliminary annual payrolls benchmark revi...,NaN,1,2024-08-21,2024-08-21,0,0,0
8,BLS to Release Oct.-Nov. Jobs Data; No Oct. Un...,NaN,1,2025-12-16,2025-12-16,0,0,0
9,CPI Index NSA,CPURNSA Index,198,2010-01-15,2026-07-14,198,198,198


# explanatio for part 6.2 
this coverage table confirms that the seven series we selected are the right primary Bloomberg series for the project.
The key coverage is very strong. CPI MoM has 197 observations and all 197 have valid consensus/actual surprises; Core CPI MoM also has 197/197; NFP has 199/199; unemployment has 198/198; Average Hourly Earnings MoM has 197 usable surprises out of 198 observations; and Core PCE has 198/198. The only notable difference is headline PCE Price Index MoM, which begins in March 2012 in this Bloomberg export and has 172 valid observations.
This is actually a very clean result. It means we can now freeze the primary series definition instead of continuing to search.
Use these exact Bloomberg series:

| Component    | Event name                    | Ticker           |
| ------------ | ----------------------------- | ---------------- |
| Headline CPI | `CPI MoM`                     | `CPI CHNG Index` |
| Core CPI     | `Core CPI MoM`                | `CPUPXCHG Index` |
| Payrolls     | `Change in Nonfarm Payrolls`  | `NFP TCH Index`  |
| Unemployment | `Unemployment Rate`           | `USURTOT Index`  |
| Wages        | `Average Hourly Earnings MoM` | `AHE MOM% Index` |
| Headline PCE | `PCE Price Index MoM`         | `PCE DEFM Index` |
| Core PCE     | `Core PCE Price Index MoM`    | `PCE CMOM Index` |



The other candidates are useful for possible robustness checks later, but they should not be part of the main specification. For example, CPI YoY and Core CPI YoY would largely duplicate information arriving in the same CPI release, and private/manufacturing payrolls would overlap with headline NFP.

In [51]:
### 6.3 Select the Final Primary Macroeconomic Series


### 6.3 Select the Final Primary Macroeconomic Series

Based on the Bloomberg coverage analysis, the main specification retains seven
macroeconomic release components with long histories and strong consensus
forecast coverage:

- CPI MoM
- Core CPI MoM
- Change in Nonfarm Payrolls
- Unemployment Rate
- Average Hourly Earnings MoM
- PCE Price Index MoM
- Core PCE Price Index MoM

The corresponding Bloomberg tickers are used to define the series
unambiguously and consistently through time.

In [53]:
# ------------------------------------------------------------
# Part 6.3: Define final primary macro series
# ------------------------------------------------------------

PRIMARY_SERIES = {
    "CPI CHNG Index": {
        "event_family": "CPI",
        "component": "headline_cpi",
    },
    
    "CPUPXCHG Index": {
        "event_family": "CPI",
        "component": "core_cpi",
    },
    
    "NFP TCH Index": {
        "event_family": "NFP",
        "component": "nonfarm_payrolls",
    },
    
    "USURTOT Index": {
        "event_family": "NFP",
        "component": "unemployment_rate",
    },
    
    "AHE MOM% Index": {
        "event_family": "NFP",
        "component": "avg_hourly_earnings",
    },
    
    "PCE DEFM Index": {
        "event_family": "PCE",
        "component": "headline_pce",
    },
    
    "PCE CMOM Index": {
        "event_family": "PCE",
        "component": "core_pce",
    },
}

primary_tickers = list(PRIMARY_SERIES.keys())

print("Number of primary series:", len(primary_tickers))
print(primary_tickers)

Number of primary series: 7
['CPI CHNG Index', 'CPUPXCHG Index', 'NFP TCH Index', 'USURTOT Index', 'AHE MOM% Index', 'PCE DEFM Index', 'PCE CMOM Index']


In [54]:
# Keep only primary macro releases


In [55]:
# ------------------------------------------------------------
# Keep only primary macro releases
# ------------------------------------------------------------

macro_surprises = (
    macro_all[
        macro_all["bloomberg_ticker"].isin(primary_tickers)
    ]
    .copy()
)

macro_surprises["event_family"] = (
    macro_surprises["bloomberg_ticker"]
    .map(lambda x: PRIMARY_SERIES[x]["event_family"])
)

macro_surprises["component"] = (
    macro_surprises["bloomberg_ticker"]
    .map(lambda x: PRIMARY_SERIES[x]["component"])
)

# Recalculate raw surprise explicitly
macro_surprises["raw_surprise"] = (
    macro_surprises["actual"]
    - macro_surprises["survey_median"]
)

macro_surprises = (
    macro_surprises
    .sort_values(
        ["release_date", "release_time",
         "event_family", "component"]
    )
    .reset_index(drop=True)
)

print("Primary macro observations:", len(macro_surprises))
print(
    "Valid raw surprises:",
    macro_surprises["raw_surprise"].notna().sum()
)

display(
    macro_surprises[
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "event_name",
            "bloomberg_ticker",
            "survey_median",
            "actual",
            "raw_surprise",
        ]
    ].head(20)
)

Primary macro observations: 1359
Valid raw surprises: 1358


,release_date,release_time,event_family,component,event_name,bloomberg_ticker,survey_median,actual,raw_surprise
0,2010-01-08,08:30:00,NFP,nonfarm_payrolls,Change in Nonfarm Payrolls,NFP TCH Index,0.0,-85.0,-85.0
1,2010-01-08,08:30:00,NFP,unemployment_rate,Unemployment Rate,USURTOT Index,10.0,10.0,0.0
2,2010-01-15,08:30:00,CPI,core_cpi,Core CPI MoM,CPUPXCHG Index,0.1,0.1,0.0
3,2010-01-15,08:30:00,CPI,headline_cpi,CPI MoM,CPI CHNG Index,0.2,0.1,-0.1
4,2010-02-01,08:30:00,PCE,core_pce,Core PCE Price Index MoM,PCE CMOM Index,0.1,0.1,0.0
5,2010-02-05,08:30:00,NFP,avg_hourly_earnings,Average Hourly Earnings MoM,AHE MOM% Index,NaN,0.2,NaN
6,2010-02-05,08:30:00,NFP,nonfarm_payrolls,Change in Nonfarm Payrolls,NFP TCH Index,15.0,-20.0,-35.0
7,2010-02-05,08:30:00,NFP,unemployment_rate,Unemployment Rate,USURTOT Index,10.0,9.7,-0.3
8,2010-02-19,08:30:00,CPI,core_cpi,Core CPI MoM,CPUPXCHG Index,0.1,-0.1,-0.2
9,2010-02-19,08:30:00,CPI,headline_cpi,CPI MoM,CPI CHNG Index,0.3,0.2,-0.1


In [56]:
### 6.4 Causal Standardization of Primary Macroeconomic Surprises


### 6.4 Causal Standardization of Primary Macroeconomic Surprises

Raw surprises are measured in different units. For example, Nonfarm Payroll
surprises are measured in thousands of jobs, while CPI and PCE surprises are
measured in percentage points.

To make the magnitude of surprises comparable across macroeconomic series, each
raw surprise is standardized relative to the historical forecast-error
distribution of the same Bloomberg series.

For release \(t\),

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
},$
\]

where \(\mu_{i,<t}\) and \(\sigma_{i,<t}\) are calculated using only previous
releases of series \(i\).

A minimum of 24 prior valid releases is required. For monthly macroeconomic
indicators, this corresponds to approximately two years of historical forecast
errors.

The current observation is excluded from its own standardization, preventing
look-ahead bias.

In [58]:
# ------------------------------------------------------------
# Part 6.4: Final causal standardization
# ------------------------------------------------------------

MIN_HISTORY = 24

# Sort chronologically within each macro series
macro_surprises = (
    macro_surprises
    .sort_values(
        [
            "bloomberg_ticker",
            "release_date",
            "release_time",
        ]
    )
    .reset_index(drop=True)
)

# Historical mean of surprise using only PRIOR observations
macro_surprises["surprise_hist_mean"] = (
    macro_surprises
    .groupby("bloomberg_ticker")["raw_surprise"]
    .transform(
        lambda x:
        x.expanding(min_periods=MIN_HISTORY)
         .mean()
         .shift(1)
    )
)

# Historical standard deviation using only PRIOR observations
macro_surprises["surprise_hist_std"] = (
    macro_surprises
    .groupby("bloomberg_ticker")["raw_surprise"]
    .transform(
        lambda x:
        x.expanding(min_periods=MIN_HISTORY)
         .std()
         .shift(1)
    )
)

# Final standardized surprise
macro_surprises["z_surprise"] = (
    macro_surprises["raw_surprise"]
    - macro_surprises["surprise_hist_mean"]
) / macro_surprises["surprise_hist_std"]

print("Total observations:", len(macro_surprises))
print(
    "Valid raw surprises:",
    macro_surprises["raw_surprise"].notna().sum()
)
print(
    "Valid standardized surprises:",
    macro_surprises["z_surprise"].notna().sum()
)

Total observations: 1359
Valid raw surprises: 1358
Valid standardized surprises: 1190


### Interpretation of the Standardization Results

The final primary macroeconomic dataset contains **1,359 release-component observations** across seven Bloomberg series:

- CPI MoM
- Core CPI MoM
- Change in Nonfarm Payrolls
- Unemployment Rate
- Average Hourly Earnings MoM
- PCE Price Index MoM
- Core PCE Price Index MoM

Of the 1,359 observations, **1,358 have valid raw surprises**. One observation does not have a Bloomberg consensus forecast, so its raw surprise cannot be calculated.

The raw surprise is defined as:

\[$
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Consensus}_{i,t}.$
\]

To make surprises comparable across macroeconomic indicators with different units, each series is standardized separately using its own historical forecast errors.

The causal standardized surprise is:

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
},$
\]

where \(\mu_{i,<t}\) and \(\sigma_{i,<t}\) are calculated using only observations released before time \(t\).

A minimum of **24 prior valid releases** is required before a standardized surprise is calculated. Since there are seven primary series,

\[
7 \times 24 = 168
\]

observations are initially used to build the historical forecast-error distributions.

Therefore,

\[
1359 - 168 - 1 = 1190
\]

observations have valid standardized surprises.

The final results are therefore:

- **Total observations:** 1,359
- **Valid raw surprises:** 1,358
- **Valid standardized surprises:** 1,190

The missing early z-scores are intentional rather than data errors. They reflect the requirement that enough historical information must exist before a release can be standardized.

This design prevents look-ahead bias because future observations are never used to construct the surprise measure for an earlier release.

The resulting **1,190 standardized macroeconomic surprises** will be used in the next stage of the project to study the intraday response of S&P 500 futures.

In [59]:
# Inspect final standardized surprises


In [60]:
# ------------------------------------------------------------
# Inspect final standardized surprises
# ------------------------------------------------------------

display(
    macro_surprises[
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "survey_median",
            "actual",
            "raw_surprise",
            "surprise_hist_mean",
            "surprise_hist_std",
            "z_surprise",
        ]
    ]
    .dropna(subset=["z_surprise"])
    .sort_values("release_date")
    .head(30)
)

,release_date,release_time,event_family,component,survey_median,actual,raw_surprise,surprise_hist_mean,surprise_hist_std,z_surprise
616,2012-01-06,08:30:00,NFP,nonfarm_payrolls,155.0,200.0,45.0,-21.687500,66.394705,1.004410
1185,2012-01-06,08:30:00,NFP,unemployment_rate,8.7,8.5,-0.2,-0.075000,0.189393,-0.660005
419,2012-01-19,08:30:00,CPI,core_cpi,0.1,0.1,0.0,-0.010833,0.085664,0.126462
222,2012-01-19,08:30:00,CPI,headline_cpi,0.1,0.0,-0.1,-0.003333,0.115370,-0.837887
815,2012-01-30,08:30:00,PCE,core_pce,0.1,0.2,0.1,-0.022792,0.058882,2.085400
617,2012-02-03,08:30:00,NFP,nonfarm_payrolls,140.0,243.0,103.0,-19.020000,66.351099,1.839005
1186,2012-02-03,08:30:00,NFP,unemployment_rate,8.5,8.3,-0.2,-0.080000,0.187083,-0.641427
420,2012-02-17,08:30:00,CPI,core_cpi,0.2,0.2,0.0,-0.010400,0.083889,0.123974
223,2012-02-17,08:30:00,CPI,headline_cpi,0.3,0.2,-0.1,-0.007200,0.114583,-0.809891
816,2012-03-01,08:30:00,PCE,core_pce,0.2,0.2,0.0,-0.017880,0.062655,0.285371


### Interpretation of the Standardized Surprise Sample

The standardized surprise table confirms that the causal surprise construction
is working as intended.

For each macroeconomic release,

\[$
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Consensus}_{i,t}.$
\]

The raw surprise is then standardized using only the historical forecast errors
of the same Bloomberg series that were available before the current release:

\[$
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
}.$
\]

For example, Nonfarm Payrolls on January 6, 2012 had a Bloomberg consensus
forecast of 155 thousand jobs and an actual release of 200 thousand jobs:

\[
200-155=45.
\]

The historical mean NFP surprise before that announcement was approximately
\(-21.69\) thousand jobs and the historical standard deviation was approximately
\(66.39\) thousand jobs. Therefore,

\[$
z=
\frac{45-(-21.69)}{66.39}
\approx
1.00.$
\]

This means the payroll release was approximately one historical standard
deviation stronger than expected.

On February 3, 2012, the NFP surprise was larger:

\[
243-140=103,
\]

producing

\[$
z\approx1.84.$
\]

Thus, the February payroll release represented a substantially more unusual
positive forecast error.

Standardization is especially useful when comparing different macroeconomic
indicators. For example, Core PCE on January 30, 2012 had a raw surprise of only

\[
0.2-0.1=0.1
\]

percentage point, but its standardized surprise was approximately

\[
z=2.09.
\]

Although the raw numerical difference appears small, it was large relative to
the historical distribution of Core PCE forecast errors.

The sign of the standardized surprise should not automatically be interpreted
as positive or negative news for equities. For example, a negative unemployment
surprise means that the unemployment rate was below consensus, which may
represent stronger labor-market conditions. Similarly, a positive inflation
surprise may be interpreted negatively by equity markets if it raises expected
interest rates.

Therefore, the surprise variables are kept in their mechanical Bloomberg form:

\[$
\text{Actual}-\text{Consensus},$
\]

and the subsequent ES futures analysis will estimate how markets actually
respond to these surprises rather than imposing a predetermined bullish or
bearish interpretation.

The first valid standardized observations appear around 2012 because each
series requires at least 24 prior valid releases before its first z-score is
calculated. These early missing z-scores are intentional and prevent unstable
standardization and look-ahead bias.

In [61]:
# Coverage by primary macro series


In [63]:
# ------------------------------------------------------------
# Coverage by primary macro series
# ------------------------------------------------------------

final_coverage = (
    macro_surprises
    .groupby(
        [
            "event_family",
            "component",
            "bloomberg_ticker",
        ]
    )
    .agg(
        n_total=("release_date", "size"),
        first_release=("release_date", "min"),
        last_release=("release_date", "max"),
        n_raw_surprise=("raw_surprise", "count"),
        n_z_surprise=("z_surprise", "count"),
        mean_z=("z_surprise", "mean"),
        std_z=("z_surprise", "std"),
        min_z=("z_surprise", "min"),
        max_z=("z_surprise", "max"),
    )
    .reset_index()
)

display(final_coverage)

,event_family,component,bloomberg_ticker,n_total,first_release,last_release,n_raw_surprise,n_z_surprise,mean_z,std_z,min_z,max_z
0,CPI,core_cpi,CPUPXCHG Index,197,2010-01-15,2026-07-14,197,173,0.049376,1.273812,-3.610079,6.492401
1,CPI,headline_cpi,CPI CHNG Index,197,2010-01-15,2026-07-14,197,173,0.092756,1.112534,-2.783518,5.998742
2,NFP,avg_hourly_earnings,AHE MOM% Index,198,2010-02-05,2026-07-02,197,173,0.245964,2.476259,-4.780868,30.715283
3,NFP,nonfarm_payrolls,NFP TCH Index,199,2010-01-08,2026-07-02,199,175,0.496465,5.165875,-9.584675,64.575553
4,NFP,unemployment_rate,USURTOT Index,198,2010-01-08,2026-07-02,198,174,-0.053575,2.516792,-30.298793,4.643547
5,PCE,core_pce,PCE CMOM Index,198,2010-02-01,2026-06-25,198,174,0.129642,1.091995,-3.121483,3.685173
6,PCE,headline_pce,PCE DEFM Index,172,2012-03-30,2026-06-25,172,148,0.168304,1.036340,-2.983889,3.609751


### Interpretation of Primary-Series Coverage

The final coverage statistics confirm that the causal standardization procedure
is working consistently across the seven primary macroeconomic series.

For each series, the first 24 valid raw surprises are used only to estimate the
initial historical forecast-error distribution. Therefore, the number of valid
standardized surprises is generally

\[$
N_z = N_{\text{raw}} - 24.$
\]

For example, Core CPI contains 197 valid raw surprises and 173 standardized
surprises:

\[$
197 - 24 = 173.$
\]

Similarly, Nonfarm Payrolls contains 199 raw surprises and 175 standardized
surprises:

\[$
199 - 24 = 175.$
\]

Headline PCE has a shorter Bloomberg history, beginning in 2012, but still
provides 172 raw surprises and 148 standardized observations.

The CPI and PCE standardized surprise distributions are reasonably centered
around zero and generally have standard deviations close to one.

However, several labor-market series contain extremely large standardized
surprises. In particular, Nonfarm Payrolls, Average Hourly Earnings, and the
Unemployment Rate contain unusually large minimum or maximum z-scores.

These observations do not necessarily indicate a coding error. Because the
standardization is causal, an unprecedented economic shock is evaluated
relative only to the forecast-error distribution observed before that shock.
Large disruptions, such as those associated with the COVID-19 period, can
therefore generate very large standardized surprises.

Nevertheless, extreme z-scores may exert excessive influence on subsequent
regressions and trading signals. Before estimating the ES futures response,
the dates and underlying raw surprises associated with these extreme values
should therefore be examined explicitly.

### Part 6.4B: Timestamp-safe causal standardization


In [87]:
# ------------------------------------------------------------
# Part 6.4B: Timestamp-safe causal standardization
# ------------------------------------------------------------

MIN_HISTORY = 24

# Create a temporary naive timestamp for ordering/grouping.
# Timezone conversion comes later in Part 7.
macro_surprises["release_datetime"] = pd.to_datetime(
    macro_surprises["release_date"].dt.strftime("%Y-%m-%d")
    + " "
    + macro_surprises["release_time"].astype(str),
    errors="coerce"
)

# ------------------------------------------------------------
# Aggregate surprise information at ticker x announcement time
# ------------------------------------------------------------

temp = macro_surprises.copy()

temp["surprise_sq"] = temp["raw_surprise"] ** 2

event_stats = (
    temp
    .groupby(
        ["bloomberg_ticker", "release_datetime"],
        as_index=False,
        dropna=False
    )
    .agg(
        event_sum=("raw_surprise", "sum"),
        event_sumsq=("surprise_sq", "sum"),
        event_count=("raw_surprise", "count"),
    )
    .sort_values(
        ["bloomberg_ticker", "release_datetime"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Historical statistics using STRICTLY EARLIER timestamps
# ------------------------------------------------------------

event_stats["hist_count"] = (
    event_stats
    .groupby("bloomberg_ticker")["event_count"]
    .transform(lambda x: x.cumsum().shift(1))
)

event_stats["hist_sum"] = (
    event_stats
    .groupby("bloomberg_ticker")["event_sum"]
    .transform(lambda x: x.cumsum().shift(1))
)

event_stats["hist_sumsq"] = (
    event_stats
    .groupby("bloomberg_ticker")["event_sumsq"]
    .transform(lambda x: x.cumsum().shift(1))
)

event_stats["surprise_hist_mean"] = (
    event_stats["hist_sum"]
    / event_stats["hist_count"]
)

# Sample variance
event_stats["surprise_hist_var"] = (
    event_stats["hist_sumsq"]
    - (
        event_stats["hist_sum"] ** 2
        / event_stats["hist_count"]
    )
) / (
    event_stats["hist_count"] - 1
)

event_stats["surprise_hist_std"] = np.sqrt(
    event_stats["surprise_hist_var"]
)

# Require at least 24 earlier valid observations
event_stats.loc[
    event_stats["hist_count"] < MIN_HISTORY,
    ["surprise_hist_mean", "surprise_hist_std"]
] = np.nan

In [88]:
# Merge timestamp-safe history back to release-level data


In [89]:
# ------------------------------------------------------------
# Merge timestamp-safe history back to release-level data
# ------------------------------------------------------------

macro_surprises = (
    macro_surprises
    .drop(
        columns=[
            "surprise_hist_mean",
            "surprise_hist_std",
            "z_surprise",
        ],
        errors="ignore"
    )
    .merge(
        event_stats[
            [
                "bloomberg_ticker",
                "release_datetime",
                "surprise_hist_mean",
                "surprise_hist_std",
                "hist_count",
            ]
        ],
        on=[
            "bloomberg_ticker",
            "release_datetime",
        ],
        how="left"
    )
)

macro_surprises["z_surprise"] = (
    macro_surprises["raw_surprise"]
    - macro_surprises["surprise_hist_mean"]
) / macro_surprises["surprise_hist_std"]

# Robustness version
macro_surprises["z_surprise_capped"] = (
    macro_surprises["z_surprise"]
    .clip(-5, 5)
)

print("Total observations:", len(macro_surprises))
print(
    "Valid raw surprises:",
    macro_surprises["raw_surprise"].notna().sum()
)
print(
    "Valid timestamp-safe z-surprises:",
    macro_surprises["z_surprise"].notna().sum()
)

Total observations: 1359
Valid raw surprises: 1358
Valid timestamp-safe z-surprises: 1190


In [90]:
display(
    macro_surprises[
        macro_surprises["multi_period_release"]
    ][
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "period",
            "raw_surprise",
            "hist_count",
            "surprise_hist_mean",
            "surprise_hist_std",
            "z_surprise",
        ]
    ]
    .sort_values(
        ["release_date", "component", "period"]
    )
)


,release_date,release_time,event_family,component,period,raw_surprise,hist_count,surprise_hist_mean,surprise_hist_std,z_surprise
901,2019-04-29,08:30:00,PCE,core_pce,Feb,0.00,110.0,-0.018473,0.057621,0.320591
902,2019-04-29,08:30:00,PCE,core_pce,Mar,-0.10,110.0,-0.018473,0.057621,-1.414892
1073,2019-04-29,08:30:00,PCE,headline_pce,Feb,0.00,84.0,-0.021131,0.059173,0.357106
1074,2019-04-29,08:30:00,PCE,headline_pce,Mar,-0.10,84.0,-0.021131,0.059173,-1.332861
190,2025-12-16,08:30:00,NFP,avg_hourly_earnings,Nov,-0.20,188.0,0.000372,0.387574,-0.516992
189,2025-12-16,08:30:00,NFP,avg_hourly_earnings,Oct,0.10,188.0,0.000372,0.387574,0.257055
783,2025-12-16,08:30:00,NFP,nonfarm_payrolls,Nov,14.00,190.0,68.731579,751.378114,-0.072842
782,2025-12-16,08:30:00,NFP,nonfarm_payrolls,Oct,-80.00,190.0,68.731579,751.378114,-0.197945
982,2026-01-22,10:00:00,PCE,core_pce,Nov,0.00,190.0,-0.012053,0.066951,0.180021
981,2026-01-22,10:00:00,PCE,core_pce,Oct,0.00,190.0,-0.012053,0.066951,0.180021


### 6.4B Correction for Simultaneous Multi-Period Releases

A later timestamp validation identified several announcements in which multiple
reference periods of the same Bloomberg series were released simultaneously.

Because these observations became public at the same time, one observation from
the announcement must not enter the historical distribution used to standardize
another observation from the same timestamp.

The causal standardization is therefore updated so that the historical mean and
standard deviation use only observations from strictly earlier announcement
timestamps.

In [65]:
# Part 6.5 — Inspect extreme standardized surprises


In [91]:
# ------------------------------------------------------------
# Part 6.5: Inspect extreme standardized surprises
# ------------------------------------------------------------

extreme_surprises = (
    macro_surprises.loc[
        macro_surprises["z_surprise"].abs() >= 5,
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "event_name",
            "survey_median",
            "actual",
            "raw_surprise",
            "surprise_hist_mean",
            "surprise_hist_std",
            "z_surprise",
        ]
    ]
    .sort_values(
        "z_surprise",
        key=lambda x: x.abs(),
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Number of observations with |z-surprise| >= 5:",
    len(extreme_surprises)
)

display(extreme_surprises)


Number of observations with |z-surprise| >= 5: 8


,release_date,release_time,event_family,component,event_name,survey_median,actual,raw_surprise,surprise_hist_mean,surprise_hist_std,z_surprise
0,2020-06-05,08:30:00,NFP,nonfarm_payrolls,Change in Nonfarm Payrolls,-7500.0,2509.0,10009.0,6.004000,154.903762,64.575553
1,2020-05-08,08:30:00,NFP,avg_hourly_earnings,Average Hourly Earnings MoM,0.4,4.7,4.3,-0.044918,0.141458,30.715283
2,2020-06-05,08:30:00,NFP,unemployment_rate,Unemployment Rate,19.0,13.3,-5.7,-0.046000,0.186608,-30.298793
3,2020-05-08,08:30:00,NFP,nonfarm_payrolls,Change in Nonfarm Payrolls,-22000.0,-20537.0,1463.0,-5.745968,82.415009,17.821341
4,2020-04-03,08:30:00,NFP,nonfarm_payrolls,Change in Nonfarm Payrolls,-100.0,-701.0,-601.0,-0.906504,62.609686,-9.584675
5,2020-05-08,08:30:00,NFP,unemployment_rate,Unemployment Rate,16.0,14.7,-1.3,-0.035887,0.149057,-8.480751
6,2021-05-12,08:30:00,CPI,core_cpi,Core CPI MoM,0.3,0.9,0.6,-0.017353,0.095089,6.492401
7,2021-05-12,08:30:00,CPI,headline_cpi,CPI MoM,0.2,0.8,0.6,-0.011250,0.101896,5.998742


### Interpretation of Extreme Standardized Surprises

**The extreme-surprise diagnostic identifies eight observations with
\[$
|z| \geq 5.$
\]
Most of these observations occur during the COVID-19 labor-market disruption
of 2020, with an additional large inflation surprise in May 2021.**

For example, on June 5, 2020, Bloomberg consensus expected Nonfarm Payrolls to
decline by approximately 7.5 million jobs, while the actual release reported an
increase of approximately 2.5 million jobs. The resulting forecast error was

\[$
2{,}509 - (-7{,}500)
=
10{,}009$
\]

thousand jobs.

Relative to the historical NFP forecast-error standard deviation available
before the release, this generated a standardized surprise of approximately

\[
z=64.6.
\]

Similarly, the June 2020 unemployment-rate release produced a standardized
surprise of approximately \(-30.3\), while the May 2020 Average Hourly Earnings
release produced a standardized surprise above \(+30\).

These observations appear to represent genuine macroeconomic shocks rather
than data-processing errors. Their unusually large z-scores arise because the
causal standardization evaluates each release relative only to the historical
forecast-error distribution available before the event. Consequently,
unprecedented shocks are not artificially normalized using information from
future observations.

However, these extreme observations may have disproportionate influence on
linear regressions and trading signals. Therefore, the original causal
z-surprise will be preserved, while capped and regime-robust versions will
also be considered as robustness checks.

The baseline research dataset therefore retains the original surprise
information rather than deleting economically meaningful extreme events.

In [92]:
# Largest positive standardized surprises


In [93]:
# Largest positive standardized surprises

display(
    macro_surprises.nlargest(
        10,
        "z_surprise"
    )[
        [
            "release_date",
            "event_family",
            "component",
            "survey_median",
            "actual",
            "raw_surprise",
            "z_surprise",
        ]
    ]
)

# Largest negative standardized surprises

display(
    macro_surprises.nsmallest(
        10,
        "z_surprise"
    )[
        [
            "release_date",
            "event_family",
            "component",
            "survey_median",
            "actual",
            "raw_surprise",
            "z_surprise",
        ]
    ]
)

,release_date,event_family,component,survey_median,actual,raw_surprise,z_surprise
717,2020-06-05,NFP,nonfarm_payrolls,-7500.0,2509.0,10009.0,64.575553
123,2020-05-08,NFP,avg_hourly_earnings,0.4,4.7,4.3,30.715283
716,2020-05-08,NFP,nonfarm_payrolls,-22000.0,-20537.0,1463.0,17.821341
531,2021-05-12,CPI,core_cpi,0.3,0.9,0.6,6.492401
334,2021-05-12,CPI,headline_cpi,0.2,0.8,0.6,5.998742
522,2020-08-12,CPI,core_cpi,0.2,0.6,0.4,4.992108
533,2021-07-13,CPI,core_cpi,0.4,0.9,0.5,4.653767
1284,2020-04-03,NFP,unemployment_rate,3.8,4.4,0.6,4.643547
923,2021-01-29,PCE,core_pce,0.1,0.3,0.2,3.685173
1115,2022-09-30,PCE,headline_pce,0.1,0.3,0.2,3.609751


,release_date,event_family,component,survey_median,actual,raw_surprise,z_surprise
1286,2020-06-05,NFP,unemployment_rate,19.0,13.3,-5.7,-30.298793
715,2020-04-03,NFP,nonfarm_payrolls,-100.0,-701.0,-601.0,-9.584675
1285,2020-05-08,NFP,unemployment_rate,16.0,14.7,-1.3,-8.480751
124,2020-06-05,NFP,avg_hourly_earnings,1.0,-1.0,-2.0,-4.780868
482,2017-04-14,CPI,core_cpi,0.2,-0.1,-0.3,-3.610079
918,2020-08-28,PCE,core_pce,0.5,0.3,-0.2,-3.121483
1046,2016-12-22,PCE,headline_pce,0.2,0.0,-0.2,-2.983889
929,2021-07-30,PCE,core_pce,0.6,0.4,-0.2,-2.796624
285,2017-04-14,CPI,headline_cpi,0.0,-0.3,-0.3,-2.783518
702,2019-03-08,NFP,nonfarm_payrolls,180.0,20.0,-160.0,-2.577158


### Interpretation of the Largest and Smallest Standardized Surprises

**The extreme-value analysis shows that the largest standardized macroeconomic surprises are concentrated in economically unusual periods rather than being
random data-processing errors.**

**The most extreme observation occurs on June 5, 2020, when Bloomberg consensus
expected Nonfarm Payrolls to decline by approximately 7.5 million jobs, while
the actual release reported an increase of approximately 2.5 million jobs:**

\[$
2{,}509 - (-7{,}500)
=
10{,}009$
\]
thousand jobs.

Relative to the historical forecast-error distribution available before the
announcement, this corresponds to a causal standardized surprise of
\[$
z \approx 64.6.$
\]

**Several other very large standardized surprises occur during the COVID-19
labor-market disruption. For example, the May 2020 Average Hourly Earnings
release has \(z \approx 30.7\), while the May 2020 NFP release has
\(z \approx 17.8\).**

The largest negative standardized surprise also occurs during this period.
On June 5, 2020, the unemployment rate was 13.3%, compared with a Bloomberg
consensus forecast of 19.0%:
\[$
13.3-19.0=-5.7,$
\]
producing
\[$
z \approx -30.3.$
\]

**Importantly, the sign of the raw or standardized surprise should not
automatically be interpreted as good or bad economic news. In this example,
the negative unemployment-rate surprise indicates that unemployment was
substantially lower than expected.**

**Large inflation surprises appear later in the sample. On May 12, 2021,
headline and core CPI both exceeded consensus by approximately 0.6 percentage
point, generating standardized surprises of approximately \(+6.0\) and
\(+6.5\), respectively.**

**These results reveal distinct macroeconomic regimes. The largest 2020
surprises are dominated by unprecedented labor-market disruptions, whereas
large 2021 surprises are increasingly associated with inflation.**

Because these extreme observations represent economically meaningful shocks,
they are retained in the baseline dataset rather than deleted. However, their
large magnitude may exert disproportionate influence on regression estimates
and trading signals. Therefore, robustness analysis will also use a capped
version of the standardized surprise while preserving the original causal
z-surprise as the main research variable.

## 7. Construct Final Macro Surprise Dataset

The final step converts the cleaned Bloomberg releases into a research-ready
event dataset for the intraday ES futures analysis.

For each macroeconomic release component, the final dataset contains:

- Eastern Time release date and time
- UTC announcement timestamp
- Macro event family
- Release component
- Bloomberg event name and ticker
- Actual release
- Bloomberg consensus forecast
- Prior value and revision
- Raw surprise
- Causal standardized surprise
- Capped standardized surprise for robustness analysis

Release timestamps are first interpreted in the
`America/New_York` timezone and are then converted to UTC.

Using an official timezone rather than manually adding four or five hours
automatically accounts for U.S. daylight-saving-time changes.

Multiple macroeconomic components may share the same timestamp. For example,
an NFP announcement can simultaneously contain payroll, unemployment, and wage
surprises. These observations are intentionally retained as separate rows.

In [98]:
# ------------------------------------------------------------
# Part 7.1: Validate release times
# ------------------------------------------------------------

# Total observations
print("Total macro observations:", len(macro_surprises))

# Check missing release times
missing_release_times = macro_surprises["release_time"].isna().sum()

print(
    "\nMissing release times:",
    missing_release_times
)

# Distribution of release times
release_time_distribution = (
    macro_surprises["release_time"]
    .value_counts(dropna=False)
    .rename("n_observations")
    .to_frame()
)

print("\nRelease-time distribution:")
display(release_time_distribution)

Total macro observations: 1359

Missing release times: 0

Release-time distribution:


,n_observations
release_time,
08:30:00,1333
10:00:00,26


In [99]:
# Then add this immediately after it to inspect non-08:30 releases:


In [100]:
# ------------------------------------------------------------
# Part 7.1A: Inspect non-08:30 release times
# ------------------------------------------------------------

non_830 = (
    macro_surprises.loc[
        macro_surprises["release_time"].astype(str) != "08:30:00",
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "period",
            "event_name",
            "bloomberg_ticker",
            "survey_median",
            "actual",
            "raw_surprise",
        ],
    ]
    .sort_values(
        [
            "release_date",
            "event_family",
            "component",
            "period",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Number of non-08:30 observations:",
    len(non_830)
)

display(non_830)


Number of non-08:30 observations: 26


,release_date,release_time,event_family,component,period,event_name,bloomberg_ticker,survey_median,actual,raw_surprise
0,2014-12-23,10:00:00,PCE,core_pce,Nov,Core PCE Price Index MoM,PCE CMOM Index,0.1000,0.0,-0.1000
1,2014-12-23,10:00:00,PCE,headline_pce,Nov,PCE Price Index MoM,PCE DEFM Index,-0.1500,-0.2,-0.0500
2,2016-02-26,10:00:00,PCE,core_pce,Jan,Core PCE Price Index MoM,PCE CMOM Index,0.2000,0.3,0.1000
3,2016-02-26,10:00:00,PCE,headline_pce,Jan,PCE Price Index MoM,PCE DEFM Index,0.0000,0.1,0.1000
4,2016-12-22,10:00:00,PCE,core_pce,Nov,Core PCE Price Index MoM,PCE CMOM Index,0.1000,0.0,-0.1000
5,2016-12-22,10:00:00,PCE,headline_pce,Nov,PCE Price Index MoM,PCE DEFM Index,0.2000,0.0,-0.2000
6,2018-12-21,10:00:00,PCE,core_pce,Nov,Core PCE Price Index MoM,PCE CMOM Index,0.2000,0.1,-0.1000
7,2018-12-21,10:00:00,PCE,headline_pce,Nov,PCE Price Index MoM,PCE DEFM Index,0.0000,0.1,0.1000
8,2019-11-27,10:00:00,PCE,core_pce,Oct,Core PCE Price Index MoM,PCE CMOM Index,0.1050,0.1,-0.0050
9,2019-11-27,10:00:00,PCE,headline_pce,Oct,PCE Price Index MoM,PCE DEFM Index,0.2745,0.2,-0.0745


In [101]:
# Part 7.1B: Summarize non-08:30 releases


In [102]:
# ------------------------------------------------------------
# Part 7.1B: Summarize non-08:30 releases
# ------------------------------------------------------------

non_830_summary = (
    non_830
    .groupby(
        [
            "event_family",
            "component",
            "event_name",
            "bloomberg_ticker",
            "release_time",
        ],
        dropna=False
    )
    .agg(
        n_observations=("release_date", "size"),
        first_date=("release_date", "min"),
        last_date=("release_date", "max"),
    )
    .reset_index()
)

display(non_830_summary)


,event_family,component,event_name,bloomberg_ticker,release_time,n_observations,first_date,last_date
0,PCE,core_pce,Core PCE Price Index MoM,PCE CMOM Index,10:00:00,13,2014-12-23,2026-01-22
1,PCE,headline_pce,PCE Price Index MoM,PCE DEFM Index,10:00:00,13,2014-12-23,2026-01-22


In [103]:
# Part 7.1C: Identify multi-period releases


In [104]:
# ------------------------------------------------------------
# Part 7.1C: Identify multi-period releases
# ------------------------------------------------------------

macro_surprises["multi_period_release"] = (
    macro_surprises.duplicated(
        subset=[
            "release_date",
            "release_time",
            "bloomberg_ticker",
        ],
        keep=False
    )
)

multi_period_events = (
    macro_surprises.loc[
        macro_surprises["multi_period_release"],
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "period",
            "event_name",
            "bloomberg_ticker",
            "raw_surprise",
            "z_surprise",
        ]
    ]
    .sort_values(
        [
            "release_date",
            "event_family",
            "component",
            "period",
        ]
    )
)

print(
    "Number of multi-period rows:",
    len(multi_period_events)
)

display(multi_period_events)

Number of multi-period rows: 12


,release_date,release_time,event_family,component,period,event_name,bloomberg_ticker,raw_surprise,z_surprise
901,2019-04-29,08:30:00,PCE,core_pce,Feb,Core PCE Price Index MoM,PCE CMOM Index,0.00,0.320591
902,2019-04-29,08:30:00,PCE,core_pce,Mar,Core PCE Price Index MoM,PCE CMOM Index,-0.10,-1.414892
1073,2019-04-29,08:30:00,PCE,headline_pce,Feb,PCE Price Index MoM,PCE DEFM Index,0.00,0.357106
1074,2019-04-29,08:30:00,PCE,headline_pce,Mar,PCE Price Index MoM,PCE DEFM Index,-0.10,-1.332861
190,2025-12-16,08:30:00,NFP,avg_hourly_earnings,Nov,Average Hourly Earnings MoM,AHE MOM% Index,-0.20,-0.516992
189,2025-12-16,08:30:00,NFP,avg_hourly_earnings,Oct,Average Hourly Earnings MoM,AHE MOM% Index,0.10,0.257055
783,2025-12-16,08:30:00,NFP,nonfarm_payrolls,Nov,Change in Nonfarm Payrolls,NFP TCH Index,14.00,-0.072842
782,2025-12-16,08:30:00,NFP,nonfarm_payrolls,Oct,Change in Nonfarm Payrolls,NFP TCH Index,-80.00,-0.197945
982,2026-01-22,10:00:00,PCE,core_pce,Nov,Core PCE Price Index MoM,PCE CMOM Index,0.00,0.180021
981,2026-01-22,10:00:00,PCE,core_pce,Oct,Core PCE Price Index MoM,PCE CMOM Index,0.00,0.180021


### Interpretation of Multi-Period Releases

The validation identifies 12 observations associated with simultaneous
multi-period announcements.

These observations correspond to three unusual release episodes:

- April 29, 2019: February and March PCE data released together
- December 16, 2025: October and November employment data released together
- January 22, 2026: October and November PCE data released together

These rows are not treated as duplicates because they correspond to different
reference periods. However, because they became public at the same announcement
time, their standardized surprises were constructed using only information
available strictly before the shared timestamp.

The observations are retained and flagged using `multi_period_release` so they
can be handled explicitly in the later ES futures event-study analysis.

### 7.2 Construct Eastern Time Announcement Timestamps

The Bloomberg dataset provides the announcement date and announcement time in
separate fields.

These fields are combined into a single timestamp and interpreted in the
`America/New_York` timezone.

Using the official U.S. Eastern Time timezone is important because it
automatically handles daylight-saving-time changes. Therefore, releases that
occur during Eastern Standard Time (EST) and Eastern Daylight Time (EDT) are
converted correctly.

Historical announcement times are preserved exactly as reported rather than
forcing all macroeconomic releases to occur at 08:30 ET.

In [105]:
# ------------------------------------------------------------
# Part 7.2: Construct Eastern Time announcement timestamp
# ------------------------------------------------------------

# Combine release date and release time into one naive datetime
macro_surprises["release_datetime_naive"] = pd.to_datetime(
    macro_surprises["release_date"].dt.strftime("%Y-%m-%d")
    + " "
    + macro_surprises["release_time"].astype(str),
    errors="coerce"
)

# Interpret the naive timestamp as U.S. Eastern Time
macro_surprises["timestamp_et"] = (
    macro_surprises["release_datetime_naive"]
    .dt.tz_localize(
        "America/New_York",
        ambiguous="raise",
        nonexistent="raise"
    )
)

print(
    "Missing Eastern Time timestamps:",
    macro_surprises["timestamp_et"].isna().sum()
)

display(
    macro_surprises[
        [
            "release_date",
            "release_time",
            "event_family",
            "component",
            "period",
            "timestamp_et",
        ]
    ]
    .sort_values("timestamp_et")
    .head(25)
)

Missing Eastern Time timestamps: 0


,release_date,release_time,event_family,component,period,timestamp_et
592,2010-01-08,08:30:00,NFP,nonfarm_payrolls,Dec,2010-01-08 08:30:00-05:00
1161,2010-01-08,08:30:00,NFP,unemployment_rate,Dec,2010-01-08 08:30:00-05:00
395,2010-01-15,08:30:00,CPI,core_cpi,Dec,2010-01-15 08:30:00-05:00
198,2010-01-15,08:30:00,CPI,headline_cpi,Dec,2010-01-15 08:30:00-05:00
791,2010-02-01,08:30:00,PCE,core_pce,Dec,2010-02-01 08:30:00-05:00
593,2010-02-05,08:30:00,NFP,nonfarm_payrolls,Jan,2010-02-05 08:30:00-05:00
1162,2010-02-05,08:30:00,NFP,unemployment_rate,Jan,2010-02-05 08:30:00-05:00
0,2010-02-05,08:30:00,NFP,avg_hourly_earnings,Jan,2010-02-05 08:30:00-05:00
199,2010-02-19,08:30:00,CPI,headline_cpi,Jan,2010-02-19 08:30:00-05:00
396,2010-02-19,08:30:00,CPI,core_cpi,Jan,2010-02-19 08:30:00-05:00


### 7.3 Convert Eastern Time Announcement Timestamps to UTC

The ES futures data are aligned using UTC timestamps.

The timezone-aware Eastern Time announcement timestamps are therefore converted
to UTC.

Because `timestamp_et` already uses the `America/New_York` timezone, the
conversion automatically accounts for daylight-saving time.

For example:

- 08:30 EST corresponds to 13:30 UTC.
- 08:30 EDT corresponds to 12:30 UTC.

This ensures that each macroeconomic announcement is matched to the correct
minute in the ES futures dataset.

In [72]:
# 7.3 Convert timestamps to UTC


In [106]:
# ------------------------------------------------------------
# Part 7.3: Convert Eastern Time timestamps to UTC
# ------------------------------------------------------------

macro_surprises["timestamp_utc"] = (
    macro_surprises["timestamp_et"]
    .dt.tz_convert("UTC")
)

# Convenient Eastern-Time date and time fields
macro_surprises["date_et"] = (
    macro_surprises["timestamp_et"]
    .dt.date
)

macro_surprises["time_et"] = (
    macro_surprises["timestamp_et"]
    .dt.strftime("%H:%M:%S")
)

print(
    "Missing UTC timestamps:",
    macro_surprises["timestamp_utc"].isna().sum()
)

display(
    macro_surprises[
        [
            "timestamp_et",
            "timestamp_utc",
            "date_et",
            "time_et",
            "event_family",
            "component",
            "period",
        ]
    ]
    .sort_values("timestamp_et")
    .head(25)
)

Missing UTC timestamps: 0


,timestamp_et,timestamp_utc,date_et,time_et,event_family,component,period
592,2010-01-08 08:30:00-05:00,2010-01-08 13:30:00+00:00,2010-01-08,08:30:00,NFP,nonfarm_payrolls,Dec
1161,2010-01-08 08:30:00-05:00,2010-01-08 13:30:00+00:00,2010-01-08,08:30:00,NFP,unemployment_rate,Dec
395,2010-01-15 08:30:00-05:00,2010-01-15 13:30:00+00:00,2010-01-15,08:30:00,CPI,core_cpi,Dec
198,2010-01-15 08:30:00-05:00,2010-01-15 13:30:00+00:00,2010-01-15,08:30:00,CPI,headline_cpi,Dec
791,2010-02-01 08:30:00-05:00,2010-02-01 13:30:00+00:00,2010-02-01,08:30:00,PCE,core_pce,Dec
593,2010-02-05 08:30:00-05:00,2010-02-05 13:30:00+00:00,2010-02-05,08:30:00,NFP,nonfarm_payrolls,Jan
1162,2010-02-05 08:30:00-05:00,2010-02-05 13:30:00+00:00,2010-02-05,08:30:00,NFP,unemployment_rate,Jan
0,2010-02-05 08:30:00-05:00,2010-02-05 13:30:00+00:00,2010-02-05,08:30:00,NFP,avg_hourly_earnings,Jan
199,2010-02-19 08:30:00-05:00,2010-02-19 13:30:00+00:00,2010-02-19,08:30:00,CPI,headline_cpi,Jan
396,2010-02-19 08:30:00-05:00,2010-02-19 13:30:00+00:00,2010-02-19,08:30:00,CPI,core_cpi,Jan


In [107]:
# ------------------------------------------------------------
# Validate UTC conversion
# ------------------------------------------------------------

utc_time_distribution = (
    macro_surprises["timestamp_utc"]
    .dt.strftime("%H:%M:%S")
    .value_counts()
    .rename("n_observations")
    .to_frame()
)

print("UTC announcement-time distribution:")
display(utc_time_distribution)

UTC announcement-time distribution:


,n_observations
timestamp_utc,
12:30:00,886
13:30:00,447
15:00:00,24
14:00:00,2


### Interpretation of UTC Timestamp Validation

The UTC timestamp distribution is consistent with the original Eastern Time
announcement schedule.

The 1,333 releases occurring at 08:30 ET are divided between:

- 12:30 UTC during Eastern Daylight Time (EDT), and
- 13:30 UTC during Eastern Standard Time (EST).

Similarly, the 26 releases occurring at 10:00 ET are converted to either:

- 14:00 UTC during EDT, or
- 15:00 UTC during EST.

The counts reconcile exactly:

\[
886 + 447 = 1333
\]

08:30 ET observations, and

\[
24 + 2 = 26
\]

10:00 ET observations.

Therefore,

\[
1333 + 26 = 1359,
\]

which equals the total number of macroeconomic release-component observations.

This confirms that the `America/New_York` timezone conversion correctly handles
historical daylight-saving-time changes and provides UTC timestamps suitable
for alignment with the ES one-minute futures data.

### 7.4 Build the Final Research Dataset

The cleaned Bloomberg data are now organized into a research-ready macroeconomic
surprise dataset.

The final dataset retains the original release information, standardized
surprise measures, timezone-aware announcement timestamps, and a flag for
simultaneous multi-period releases.

`event_family` is renamed to `event_type`, and `survey_median` is renamed to
`consensus` for clarity.

In [108]:
# ------------------------------------------------------------
# Part 7.4: Build final macro surprise dataset
# ------------------------------------------------------------

final_macro = macro_surprises.copy()

# Clear research-friendly names
final_macro = final_macro.rename(
    columns={
        "event_family": "event_type",
        "survey_median": "consensus",
    }
)

final_columns = [
    "timestamp_utc",
    "timestamp_et",
    "date_et",
    "time_et",
    "event_type",
    "component",
    "period",
    "event_name",
    "bloomberg_ticker",
    "actual",
    "consensus",
    "prior",
    "revision",
    "raw_surprise",
    "surprise_hist_mean",
    "surprise_hist_std",
    "z_surprise",
    "z_surprise_capped",
    "multi_period_release",
]

final_macro = (
    final_macro[final_columns]
    .sort_values(
        [
            "timestamp_utc",
            "event_type",
            "component",
            "period",
        ]
    )
    .reset_index(drop=True)
)

print("Final dataset shape:", final_macro.shape)

display(final_macro.head(20))

Final dataset shape: (1359, 19)


,timestamp_utc,timestamp_et,date_et,time_et,event_type,component,period,event_name,bloomberg_ticker,actual,consensus,prior,revision,raw_surprise,surprise_hist_mean,surprise_hist_std,z_surprise,z_surprise_capped,multi_period_release
0,2010-01-08 13:30:00+00:00,2010-01-08 08:30:00-05:00,2010-01-08,08:30:00,NFP,nonfarm_payrolls,Dec,Change in Nonfarm Payrolls,NFP TCH Index,-85.0,0.0,-11.0,-4.000000,-85.0,NaN,NaN,NaN,NaN,False
1,2010-01-08 13:30:00+00:00,2010-01-08 08:30:00-05:00,2010-01-08,08:30:00,NFP,unemployment_rate,Dec,Unemployment Rate,USURTOT Index,10.0,10.0,10.0,9.900000,0.0,NaN,NaN,NaN,NaN,False
2,2010-01-15 13:30:00+00:00,2010-01-15 08:30:00-05:00,2010-01-15,08:30:00,CPI,core_cpi,Dec,Core CPI MoM,CPUPXCHG Index,0.1,0.1,0.0,0.057599,0.0,NaN,NaN,NaN,NaN,False
3,2010-01-15 13:30:00+00:00,2010-01-15 08:30:00-05:00,2010-01-15,08:30:00,CPI,headline_cpi,Dec,CPI MoM,CPI CHNG Index,0.1,0.2,0.4,0.294263,-0.1,NaN,NaN,NaN,NaN,False
4,2010-02-01 13:30:00+00:00,2010-02-01 08:30:00-05:00,2010-02-01,08:30:00,PCE,core_pce,Dec,Core PCE Price Index MoM,PCE CMOM Index,0.1,0.1,0.0,0.106001,0.0,NaN,NaN,NaN,NaN,False
5,2010-02-05 13:30:00+00:00,2010-02-05 08:30:00-05:00,2010-02-05,08:30:00,NFP,avg_hourly_earnings,Jan,Average Hourly Earnings MoM,AHE MOM% Index,0.2,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,False
6,2010-02-05 13:30:00+00:00,2010-02-05 08:30:00-05:00,2010-02-05,08:30:00,NFP,nonfarm_payrolls,Jan,Change in Nonfarm Payrolls,NFP TCH Index,-20.0,15.0,-85.0,-249.000000,-35.0,NaN,NaN,NaN,NaN,False
7,2010-02-05 13:30:00+00:00,2010-02-05 08:30:00-05:00,2010-02-05,08:30:00,NFP,unemployment_rate,Jan,Unemployment Rate,USURTOT Index,9.7,10.0,10.0,9.900000,-0.3,NaN,NaN,NaN,NaN,False
8,2010-02-19 13:30:00+00:00,2010-02-19 08:30:00-05:00,2010-02-19,08:30:00,CPI,core_cpi,Jan,Core CPI MoM,CPUPXCHG Index,-0.1,0.1,0.1,NaN,-0.2,NaN,NaN,NaN,NaN,False
9,2010-02-19 13:30:00+00:00,2010-02-19 08:30:00-05:00,2010-02-19,08:30:00,CPI,headline_cpi,Jan,CPI MoM,CPI CHNG Index,0.2,0.3,0.1,0.093112,-0.1,NaN,NaN,NaN,NaN,False


In [109]:
# Part 7.4A: Validate final research dataset


In [110]:
# ------------------------------------------------------------
# Part 7.4A: Validate final research dataset
# ------------------------------------------------------------

print("Total rows:", len(final_macro))

print(
    "Valid raw surprises:",
    final_macro["raw_surprise"].notna().sum()
)

print(
    "Valid z-surprises:",
    final_macro["z_surprise"].notna().sum()
)

print(
    "Missing UTC timestamps:",
    final_macro["timestamp_utc"].isna().sum()
)

print(
    "Missing ET timestamps:",
    final_macro["timestamp_et"].isna().sum()
)

print(
    "Multi-period rows:",
    final_macro["multi_period_release"].sum()
)

print("\nObservations by event type:")

display(
    final_macro["event_type"]
    .value_counts()
    .rename("n_observations")
    .to_frame()
)

print("\nObservations by component:")

display(
    final_macro["component"]
    .value_counts()
    .rename("n_observations")
    .to_frame()
)

Total rows: 1359
Valid raw surprises: 1358
Valid z-surprises: 1190
Missing UTC timestamps: 0
Missing ET timestamps: 0
Multi-period rows: 12

Observations by event type:


,n_observations
event_type,
NFP,595
CPI,394
PCE,370



Observations by component:


,n_observations
component,
nonfarm_payrolls,199
unemployment_rate,198
core_pce,198
avg_hourly_earnings,198
core_cpi,197
headline_cpi,197
headline_pce,172


### Interpretation of Component Coverage

The final macroeconomic dataset contains 1,359 release-component observations
across seven primary macroeconomic series.

The component counts sum exactly to the full sample:

\[
199 + 198 + 198 + 198 + 197 + 197 + 172 = 1359.
\]

Nonfarm Payrolls has the largest number of observations, with 199 releases.
Unemployment Rate, Core PCE, and Average Hourly Earnings each contain 198
observations, while Core CPI and Headline CPI each contain 197 observations.

Headline PCE contains fewer observations, with 172 releases, because the
Bloomberg monthly headline PCE series begins later in the sample.

The component-level counts therefore reconcile correctly with the total dataset
size and confirm that the final research sample contains the intended seven
macroeconomic release components.

In [111]:
# Part 7.5: Final integrity checks


In [112]:
# ------------------------------------------------------------
# Part 7.5: Final integrity checks
# ------------------------------------------------------------

print("Final dataset shape:", final_macro.shape)

print("\nMissing-value checks:")
print("Missing timestamp_utc:",
      final_macro["timestamp_utc"].isna().sum())

print("Missing timestamp_et:",
      final_macro["timestamp_et"].isna().sum())

print("Missing raw_surprise:",
      final_macro["raw_surprise"].isna().sum())

print("Missing z_surprise:",
      final_macro["z_surprise"].isna().sum())

print("\nMulti-period rows:")
print(final_macro["multi_period_release"].sum())

print("\nExact duplicate rows:")
print(final_macro.duplicated().sum())

print("\nDuplicate timestamp + ticker + period combinations:")
print(
    final_macro.duplicated(
        subset=[
            "timestamp_utc",
            "bloomberg_ticker",
            "period",
        ]
    ).sum()
)

print("\nDate range:")
print(
    final_macro["timestamp_utc"].min(),
    "to",
    final_macro["timestamp_utc"].max()
)

Final dataset shape: (1359, 19)

Missing-value checks:
Missing timestamp_utc: 0
Missing timestamp_et: 0
Missing raw_surprise: 1
Missing z_surprise: 169

Multi-period rows:
12

Exact duplicate rows:
0

Duplicate timestamp + ticker + period combinations:
0

Date range:
2010-01-08 13:30:00+00:00 to 2026-07-14 12:30:00+00:00


### Final Integrity Check

The final macroeconomic surprise dataset passes all major data-integrity checks.

The dataset contains **1,359 release-component observations** covering the
period from January 2010 through July 2026.

There are no missing Eastern Time or UTC announcement timestamps:

- Missing `timestamp_et`: 0
- Missing `timestamp_utc`: 0

Only one observation has a missing raw surprise because Bloomberg does not
provide a consensus forecast for that release.

There are 169 missing standardized surprises. This is expected from the causal
standardization procedure. The first 24 observations of each of the seven
macroeconomic series are required to construct the initial historical
forecast-error distribution:

\[
7 \times 24 = 168.
\]

Together with the single observation that lacks a valid raw surprise,

\[
168 + 1 = 169,
\]

which exactly explains the missing standardized-surprise count.

The dataset contains 12 observations associated with simultaneous multi-period
releases. These observations are intentionally retained and identified using
the `multi_period_release` flag.

No exact duplicate rows are present, and there are no duplicate combinations
of announcement timestamp, Bloomberg ticker, and reference period.

These results confirm that the final dataset is internally consistent and
ready to be saved for the ES futures event-response analysis.

### 7.6 Save the Final Macroeconomic Surprise Dataset

The validated research dataset is saved in Parquet format for use in the
subsequent ES futures event-response analysis.

Parquet preserves data types efficiently, including timezone-aware timestamps,
and is substantially more efficient than CSV for repeated analysis.

The output file is:

`data/processed/macro_surprises_2010_2026.parquet`

In [113]:
# ------------------------------------------------------------
# Part 7.6: Save final macro surprise dataset
# ------------------------------------------------------------

OUTPUT_FILE = (
    PROCESSED_DIR
    / "macro_surprises_2010_2026.parquet"
)

final_macro.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("Saved final macro surprise dataset:")
print(OUTPUT_FILE)

print("\nFile exists:", OUTPUT_FILE.exists())

print(
    "File size (MB):",
    round(
        OUTPUT_FILE.stat().st_size / 1_000_000,
        3
    )
)

Saved final macro surprise dataset:
/Users/reza/Desktop/mfe-term 3/BlackRock-Project-State-Dependent-Intraday-Response-of-S-P-500-Futures-to-Macroeconomic-Surprises/data/processed/macro_surprises_2010_2026.parquet

File exists: True
File size (MB): 0.091


In [114]:
# Part 7.6A: Read-back validation


In [115]:
# ------------------------------------------------------------
# Part 7.6A: Read-back validation
# ------------------------------------------------------------

macro_check = pd.read_parquet(OUTPUT_FILE)

print("Saved shape:", macro_check.shape)

print(
    "Timestamp UTC dtype:",
    macro_check["timestamp_utc"].dtype
)

print(
    "Timestamp ET dtype:",
    macro_check["timestamp_et"].dtype
)

print(
    "Valid raw surprises:",
    macro_check["raw_surprise"].notna().sum()
)

print(
    "Valid z-surprises:",
    macro_check["z_surprise"].notna().sum()
)

print(
    "Multi-period rows:",
    macro_check["multi_period_release"].sum()
)

display(macro_check.head())

Saved shape: (1359, 19)
Timestamp UTC dtype: datetime64[us, UTC]
Timestamp ET dtype: datetime64[us, America/New_York]
Valid raw surprises: 1358
Valid z-surprises: 1190
Multi-period rows: 12


,timestamp_utc,timestamp_et,date_et,time_et,event_type,component,period,event_name,bloomberg_ticker,actual,consensus,prior,revision,raw_surprise,surprise_hist_mean,surprise_hist_std,z_surprise,z_surprise_capped,multi_period_release
0,2010-01-08 13:30:00+00:00,2010-01-08 08:30:00-05:00,2010-01-08,08:30:00,NFP,nonfarm_payrolls,Dec,Change in Nonfarm Payrolls,NFP TCH Index,-85.0,0.0,-11.0,-4.000000,-85.0,NaN,NaN,NaN,NaN,False
1,2010-01-08 13:30:00+00:00,2010-01-08 08:30:00-05:00,2010-01-08,08:30:00,NFP,unemployment_rate,Dec,Unemployment Rate,USURTOT Index,10.0,10.0,10.0,9.900000,0.0,NaN,NaN,NaN,NaN,False
2,2010-01-15 13:30:00+00:00,2010-01-15 08:30:00-05:00,2010-01-15,08:30:00,CPI,core_cpi,Dec,Core CPI MoM,CPUPXCHG Index,0.1,0.1,0.0,0.057599,0.0,NaN,NaN,NaN,NaN,False
3,2010-01-15 13:30:00+00:00,2010-01-15 08:30:00-05:00,2010-01-15,08:30:00,CPI,headline_cpi,Dec,CPI MoM,CPI CHNG Index,0.1,0.2,0.4,0.294263,-0.1,NaN,NaN,NaN,NaN,False
4,2010-02-01 13:30:00+00:00,2010-02-01 08:30:00-05:00,2010-02-01,08:30:00,PCE,core_pce,Dec,Core PCE Price Index MoM,PCE CMOM Index,0.1,0.1,0.0,0.106001,0.0,NaN,NaN,NaN,NaN,False


# 8- Final Results and Conclusion

This notebook constructed a research-ready dataset of U.S. macroeconomic
surprises for the BlackRock project:

**State-Dependent Intraday Response of S&P 500 Futures to Macroeconomic Surprises**

## Data Processing Summary

The original Bloomberg workbook contained:

\[
\boxed{28,159}
\]

U.S. economic-release observations covering January 2010 through July 2026.

The analysis was restricted to seven primary macroeconomic release components
from three major event families:

### CPI

- Headline CPI MoM
- Core CPI MoM

### NFP / Labor Market

- Change in Nonfarm Payrolls
- Unemployment Rate
- Average Hourly Earnings MoM

### PCE

- Headline PCE Price Index MoM
- Core PCE Price Index MoM

After filtering the Bloomberg calendar to these primary series, the final sample
contains:

\[
\boxed{1,359}
\]

release-component observations.

---

## Macroeconomic Surprise Construction

For each economic release, the raw surprise is defined as:

\[
\text{Raw Surprise}_{i,t}
=
\text{Actual}_{i,t}
-
\text{Consensus}_{i,t}.
\]

Of the 1,359 observations,

\[
\boxed{1,358}
\]

have valid raw surprises.

One observation does not have a Bloomberg consensus forecast and therefore
cannot be assigned a raw surprise.

Because the macroeconomic indicators are measured in different units, each
series was standardized separately using its own historical forecast-error
distribution.

The causal standardized surprise is:

\[
z_{i,t}
=
\frac{
\text{Raw Surprise}_{i,t}
-
\mu_{i,<t}
}{
\sigma_{i,<t}
}.
\]

Only information available strictly before the current announcement is used.

A minimum of 24 prior valid observations is required before calculating a
standardized surprise.

The final dataset therefore contains:

\[
\boxed{1,190}
\]

valid causal standardized surprises.

The 169 missing z-scores are fully explained by:

\[
7 \times 24 = 168
\]

initial observations required to establish the historical distributions, plus
one observation with a missing raw surprise:

\[
168 + 1 = 169.
\]

---

## Protection Against Look-Ahead Bias

The standardization procedure was designed to avoid look-ahead bias.

Special attention was given to announcements in which multiple reference
periods of the same economic series were released simultaneously.

Examples include:

- April 29, 2019: multiple PCE periods
- December 16, 2025: multiple employment-report periods
- January 22, 2026: multiple PCE periods

These releases were not treated as duplicates.

Instead, historical means and standard deviations were calculated using only
observations from announcement timestamps strictly earlier than the current
release.

The final dataset contains:

\[
\boxed{12}
\]

rows flagged as simultaneous multi-period releases.

---

## Extreme Macroeconomic Surprises

The largest standardized surprises are concentrated in economically unusual
periods, particularly the COVID-19 disruption.

For example, on June 5, 2020, Bloomberg expected Nonfarm Payrolls to decline by
approximately 7.5 million jobs, while the actual release reported an increase of
approximately 2.5 million jobs.

The raw surprise was:

\[$
2,509 - (-7,500)
=
10,009$
\]

thousand jobs, resulting in a very large causal standardized surprise.

Large inflation surprises also appear during the post-COVID inflation period,
particularly in CPI releases during 2021.

These observations were retained because they represent economically meaningful
shocks rather than obvious data errors.

The original `z_surprise` variable is preserved, while a capped version,
`z_surprise_capped`, is also retained for robustness analysis.

---

## Announcement Timestamp Construction

Bloomberg release dates and times were converted into timezone-aware
announcement timestamps.

The original Eastern Time release schedule contains:

- 1,333 observations released at 08:30 ET
- 26 observations released at 10:00 ET

No selected macroeconomic observation has a missing release time.

Eastern Time timestamps were constructed using:

`America/New_York`

so daylight-saving-time changes are handled automatically.

The timestamps were then converted to UTC for alignment with the ES one-minute
futures data.

The resulting UTC announcement times are distributed across:

- 12:30 UTC
- 13:30 UTC
- 14:00 UTC
- 15:00 UTC

depending on the historical Eastern Time release time and daylight-saving
status.

There are:

\[
\boxed{0}
\]

missing Eastern Time timestamps and

\[
\boxed{0}
\]

missing UTC timestamps.

---

## Final Dataset Integrity

The final research dataset has:

\[$
\boxed{1,359 \text{ rows} \times 19 \text{ columns}}$
\]

and passes the major integrity checks:

- Missing UTC timestamps: **0**
- Missing Eastern Time timestamps: **0**
- Missing raw surprises: **1**
- Missing standardized surprises: **169**
- Multi-period release rows: **12**
- Exact duplicate rows: **0**
- Duplicate timestamp/ticker/period combinations: **0**

The final macroeconomic sample covers:

\[
\boxed{
\text{January 8, 2010}
\rightarrow
\text{July 14, 2026}
}
\]

---

## Final Output

The cleaned research dataset is saved as:

`data/processed/macro_surprises_2010_2026.parquet`

This file contains the macroeconomic information required for the next stage of
the project, including:

- announcement timestamps in ET and UTC,
- event family,
- release component,
- Bloomberg ticker,
- actual value,
- consensus forecast,
- prior value,
- revision,
- raw surprise,
- causal standardized surprise,
- capped standardized surprise,
- and multi-period release flags.

## Next Step

The next notebook will merge these macroeconomic surprises with the one-minute
E-mini S&P 500 futures data.

The objective will be to measure the ES futures response over multiple
post-announcement horizons:

\[$
1,\ 5,\ 15,\ 30,\ \text{and}\ 60\text{ minutes}.$
\]

The next stage will therefore study:

\[$
\boxed{
\text{Macroeconomic Surprise}
\rightarrow
\text{Intraday ES Futures Response}
}$
\]

and subsequently test whether this response depends on the market state
immediately before the announcement.